# Instant-1.7B — Step 2: build the training data (the model's own verified answers)

**Goal:** teach `LiquidAI/LFM3-1.7B` to answer **directly** (no thinking, no hidden reasoning) while keeping the accuracy it reaches *with* thinking.

Our baseline (`results/baseline_v2`) showed: thinking OFF often still reasons in secret, and where it really answers directly it loses accuracy (MBPP+ −11, tool calls −13 points). Thinking ON is accurate but long.

**How this notebook makes the training data:**

| Step | What happens |
|---|---|
| 1. New questions | Code, math, instruction-following and tool-calling questions from openly licensed **training** sets. Nothing from any benchmark we test on; a 13-word-overlap check removes anything similar. |
| 2. Pass A (cheap) | LFM3 answers every question with thinking **OFF**. |
| 3. Pass B (only where A failed) | LFM3 answers again with thinking **ON**, 2 tries each. |
| 4. Verify | Code must pass its tests, math must hit the reference number, instructions are checked by Google's IFEval checker, tool calls must match the reference call. |
| 5. Keep the best | For each question: the **shortest correct final answer**, with any thinking removed. That is one training example: *question → direct, correct answer*. |

So the model learns from its own best answers. No other company's model is used (no licence problems).

| Source | Used for | Licence |
|---|---|---|
| MBPP (train + validation + prompt splits, minus every MBPP+ task) | Python functions with tests | CC-BY-4.0 |
| APPS (train split, introductory + interview, stdin problems) | competitive-style programs with tests | MIT |
| GSM8K (train split) | math word problems | MIT |
| OpenAssistant OASST2 prompts + IFEval-style checkable rules | instruction following | Apache-2.0 |
| Hermes function-calling v1 (single turn) | tool calls, incl. several calls at once | Apache-2.0 |

**Honesty note:** the instruction rules come from IFEval's rule *types* (with new prompts), so IFEval scores after training are partly "same skill, same rule types". **IFBench** uses different, unseen rules and is the clean test of instruction following.

**Kaggle:** Accelerator **GPU T4 x2**, Internet **On**. Rough time with the default sizes (~3,200 questions): 2–4 h, shown live while it runs. Every answer is saved at once: after a disconnect, *Run All* continues where it stopped.

## 1. Install

In [ ]:
!pip install -q -U transformers accelerate
!pip install -q absl-py langdetect immutabledict nltk

## 2. Settings
Lower the numbers in `N_PER_DOMAIN` to finish faster.

In [ ]:
import os
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

MODEL = "LiquidAI/LFM3-1.7B"
N_PER_DOMAIN = {"mbpp": None,      # None = all usable (~450)
                "apps": 700,
                "gsm8k": 700,
                "if": 700,         # instruction following
                "tools": 600,      # tool calls
                "no_tool": 100}    # tools offered but none fits: the right answer is to NOT call one
TRIES_ON = 2                # pass B: thinking-ON tries per question that pass A could not solve
MAX_NEW_A = 2048            # pass A (thinking OFF) token cap
MAX_NEW_B = 4096            # pass B (thinking ON) token cap
MAX_TARGET_TOKENS = 1536    # longest final answer we accept as a training target
BATCH = 64                  # max answers at once per GPU
BATCH_TOKENS = 80_000       # max prompt+answer tokens at once per GPU (lowers itself on "out of memory", then grows back)
SEED = 0

SAMPLING = {
    "general": dict(temperature=0.1, top_k=50, top_p=1.0, repetition_penalty=1.1),     # Liquid's default
    "diverse": dict(temperature=0.6, top_k=50, top_p=0.95, repetition_penalty=1.05),   # pass B: tries should differ
    "math":    dict(temperature=1.0, top_k=20, top_p=0.95, repetition_penalty=1.0),    # Liquid's math setting
}

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
DATA_DIR = os.path.join(WORK, "data")
OUT_DIR = os.path.join(WORK, "instant_data")
GEN_DIR = os.path.join(WORK, "gen_step2")
for d in (DATA_DIR, OUT_DIR, GEN_DIR):
    os.makedirs(d, exist_ok=True)
print("Working dir:", WORK)

## 3. Tokenizer and chat template
The notebook itself only needs the tokenizer (to build prompts). The model runs in separate GPU workers (section 8), one per GPU, so all GPU memory goes to generation.

In [ ]:
import time, json, torch
from transformers import AutoTokenizer

assert torch.cuda.is_available(), "No GPU: set Accelerator to 'GPU T4 x2'."
N_GPUS = torch.cuda.device_count()
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPUS)])

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)

def render(messages, thinking, tools=None):
    return tokenizer.apply_chat_template(messages, tools=tools, tokenize=False, add_generation_prompt=True,
                                         enable_thinking=thinking)

def n_tokens(text):
    return len(tokenizer.encode(text, add_special_tokens=False)) if text else 0

_m = [{"role": "user", "content": "What is 2+2?"}]
print("ON :", repr(render(_m, True)[-60:]))
print("OFF:", repr(render(_m, False)[-60:]))
assert render(_m, True) != render(_m, False)

## 4. Helpers
Same parsing as the benchmark notebook: split thinking from the answer, pull out code / numbers / tool calls, run programs safely with a time and memory limit.

In [ ]:
import re, ast, subprocess, sys, tempfile
from concurrent.futures import ThreadPoolExecutor

def split_thinking(raw, prompt_opened_think=False):
    # -> (reasoning, answer, status): closed / unclosed / none
    if "</think>" in raw:
        pre, post = raw.rsplit("</think>", 1)
        return pre.replace("<think>", "").strip(), post.strip(), "closed"
    if "<think>" in raw or prompt_opened_think:
        return raw.split("<think>", 1)[-1].strip(), "", "unclosed"
    return "", raw.strip(), "none"

_FENCE = re.compile(r"```[ \t]*([A-Za-z0-9_+-]*)[^\n]*\n(.*?)```", re.DOTALL)

def extract_code(answer, entry_point=None):
    blocks = [(lang.lower(), body) for lang, body in _FENCE.findall(answer)]
    if not blocks:
        m = re.search(r"```[ \t]*[A-Za-z0-9_+-]*[^\n]*\n(.*)", answer, re.DOTALL)
        return (m.group(1) if m else answer).strip("\n")
    py = [b for lang, b in blocks if lang in ("", "python", "py", "python3")] or [b for _, b in blocks]
    if entry_point:
        defining = [b for b in py if re.search(rf"\bdef\s+{re.escape(entry_point)}\s*\(", b)]
        if defining:
            return defining[-1].strip("\n")
    return max(py, key=len).strip("\n")

_NUM = re.compile(r"-?\d[\d,]*(?:\.\d+)?")

def _boxed(text):
    out, i = [], 0
    while (i := text.find("\\boxed{", i)) != -1:
        j = start = i + len("\\boxed{"); depth = 1
        while j < len(text) and depth:
            depth += {"{": 1, "}": -1}.get(text[j], 0); j += 1
        out.append(text[start:j - 1] if depth == 0 else text[start:]); i = j
    return out

def extract_final_number(text):
    boxed = _boxed(text)
    if boxed: cand = boxed[-1]
    elif "####" in text: cand = text.rsplit("####", 1)[-1]
    else:
        m = re.findall(r"answer\s*(?:is|:)\s*([^\n]+)", text, flags=re.IGNORECASE)
        cand = m[-1] if m else text
    nums = _NUM.findall(cand) or _NUM.findall(text)
    try: return float(nums[-1].replace(",", "")) if nums else None
    except ValueError: return None

# LFM3 tool calls: <tool_call>\n<function=NAME>\n<parameter=KEY>\nVALUE\n</parameter>\n</function>\n</tool_call>
_CALL = re.compile(r"<tool_call>\s*<function=([^\n>]+)>(.*?)</function>\s*</tool_call>", re.DOTALL)
_PARAM = re.compile(r"<parameter=([^>\n]+)>\n?(.*?)\n?</parameter>", re.DOTALL)

def _typed(text, schema_type):
    t = (schema_type or "").lower()
    if t in ("string", "any"):
        return text
    for parse in (json.loads, ast.literal_eval):
        try:
            return parse(text.strip())
        except Exception:
            pass
    return text

def parse_tool_calls(answer, tools=()):
    props = {t["function"]["name"]: t["function"].get("parameters", {}).get("properties", {}) for t in tools}
    calls = []
    for name, body in _CALL.findall(answer):
        name = name.strip()
        args = {k.strip(): _typed(v, props.get(name, {}).get(k.strip(), {}).get("type")) for k, v in _PARAM.findall(body)}
        calls.append({"name": name, "arguments": args})
    return calls

def _render_value(v):   # how the chat template writes a value: strings raw, everything else JSON
    if isinstance(v, str): return v
    if isinstance(v, bool): return "true" if v else "false"
    if v is None: return "null"
    return json.dumps(v) if isinstance(v, (list, dict)) else str(v)

def render_tool_calls(calls):
    return "\n".join("<tool_call>\n<function=" + c["name"] + ">\n"
                     + "".join(f"<parameter={k}>\n{_render_value(v)}\n</parameter>\n" for k, v in c["arguments"].items())
                     + "</function>\n</tool_call>" for c in calls)

def _norm(v):
    if isinstance(v, bool): return v
    if isinstance(v, (int, float)): return round(float(v), 6)
    if isinstance(v, str):
        s = v.strip()
        try: return round(float(s), 6)
        except ValueError: return s.lower()
    if isinstance(v, list): return [_norm(x) for x in v]
    if isinstance(v, dict): return {k: _norm(x) for k, x in sorted(v.items())}
    return v

def same_calls(got, ref):
    # same tool calls, same arguments, any order
    key = lambda c: json.dumps({"name": c["name"], "arguments": _norm(c["arguments"])}, sort_keys=True)
    return sorted(map(key, got)) == sorted(map(key, ref))

# ---- running programs (time + memory limit)
def _limits():
    try:
        import resource
        resource.setrlimit(resource.RLIMIT_AS, (4 << 30, 4 << 30))
    except Exception:
        pass

def run_program(src, stdin="", timeout=10):
    with tempfile.TemporaryDirectory() as d:
        path = os.path.join(d, "prog.py"); open(path, "w").write(src)
        try:
            p = subprocess.run([sys.executable, path], input=stdin, capture_output=True, text=True, timeout=timeout,
                               cwd=d, preexec_fn=_limits if os.name == "posix" else None)
            return p.returncode, p.stdout
        except subprocess.TimeoutExpired:
            return -9, ""

def same_output(got, expected):
    gl, el = [l.strip() for l in got.strip().splitlines()], [l.strip() for l in expected.strip().splitlines()]
    if gl == el: return True
    if len(gl) != len(el): return False
    for a, b in zip(gl, el):
        ta, tb = a.split(), b.split()
        if len(ta) != len(tb): return False
        for x, y in zip(ta, tb):
            if x == y: continue
            try:
                if abs(float(x) - float(y)) > 1e-6 * max(1, abs(float(y))): return False
            except ValueError:
                return False
    return True

assert split_thinking("a</think>b") == ("a", "b", "closed")
assert extract_final_number("so \\boxed{1,234}") == 1234
_t = [{"type": "function", "function": {"name": "f", "parameters": {"properties": {"n": {"type": "integer"}, "s": {"type": "string"}}}}}]
_c = [{"name": "f", "arguments": {"n": 3, "s": "Hi"}}]
assert same_calls(parse_tool_calls(render_tool_calls(_c), _t), _c)
print("helpers OK")

## 5. Download the sources (pinned versions)
Training sources, plus every benchmark we evaluate on (only to check that no training question overlaps with them).

In [ ]:
import gzip, random, shutil, urllib.request

HF = "https://huggingface.co/datasets"
SOURCES = {
    "mbpp.jsonl": "https://raw.githubusercontent.com/google-research/google-research/master/mbpp/mbpp.jsonl",
    "gsm8k_train.jsonl": "https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/train.jsonl",
    "apps_train.jsonl": f"{HF}/codeparrot/apps/resolve/21e74ddf8de1a21436da12e3e653065c5213e9d1/train.jsonl",            # ~107 MB
    "hermes_fc_singleturn.json": f"{HF}/NousResearch/hermes-function-calling-v1/resolve/dae3e1d28cfbcf4b915c04ea1e072030529b4bda/func-calling-singleturn.json",
    "oasst2_prompts.jsonl.gz": f"{HF}/OpenAssistant/oasst2/resolve/179dd21fc55192153d94adb0e0ce8f69e222bf75/2023-11-05_oasst2_prompts.messages.jsonl.gz",
}
BFCL_RAW = "https://raw.githubusercontent.com/ShishirPatil/gorilla/6ea57973c7a6097fd7c5915698c54c17c5b1b6c8/berkeley-function-call-leaderboard/bfcl_eval"
EVAL_FILES = {   # only used for the overlap check
    "HumanEvalPlus.jsonl.gz": "https://github.com/evalplus/humanevalplus_release/releases/download/v0.1.10/HumanEvalPlus.jsonl.gz",
    "MbppPlus.jsonl.gz": "https://github.com/evalplus/mbppplus_release/releases/download/v0.2.0/MbppPlus.jsonl.gz",
    "gsm8k_test.jsonl": "https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl",
    "ifeval_input_data.jsonl": "https://raw.githubusercontent.com/google-research/google-research/master/instruction_following_eval/data/input_data.jsonl",
    "ifbench_test.jsonl": "https://raw.githubusercontent.com/allenai/IFBench/cd189a9c2889526d8af901d2103fd3e1ffcea6df/data/IFBench_test.jsonl",
    "livecodebench_v6.jsonl": f"{HF}/livecodebench/code_generation_lite/resolve/0fe84c3912ea0c4d4a78037083943e8f0c4dd505/test6.jsonl",
}
for c in ["simple_python", "multiple", "parallel", "parallel_multiple", "live_simple", "live_multiple", "irrelevance"]:
    EVAL_FILES[f"BFCL_v4_{c}.json"] = f"{BFCL_RAW}/data/BFCL_v4_{c}.json"

def download(url, dest, tries=4):
    for k in range(tries):
        try:
            req = url
            if k == tries - 1 and "/releases/download/" in url:   # last try: same file via the GitHub API
                owner, repo, _, _, tag, name = url.split("github.com/")[1].split("/")
                api = json.load(urllib.request.urlopen(f"https://api.github.com/repos/{owner}/{repo}/releases/tags/{tag}", timeout=60))
                asset = next(a["url"] for a in api["assets"] if a["name"] == name)
                req = urllib.request.Request(asset, headers={"Accept": "application/octet-stream"})
            with urllib.request.urlopen(req, timeout=180) as r, open(dest + ".part", "wb") as f:
                shutil.copyfileobj(r, f)
            os.replace(dest + ".part", dest); return
        except Exception as e:
            print(f"  download problem ({e}), retrying..."); time.sleep(3 * (k + 1))
    raise RuntimeError(f"could not download {url}")

for name, url in {**SOURCES, **EVAL_FILES}.items():
    if not os.path.exists(os.path.join(DATA_DIR, name)):
        print("downloading", name); download(url, os.path.join(DATA_DIR, name))

def read_jsonl(name):
    path = os.path.join(DATA_DIR, name)
    with (gzip.open(path, "rt", encoding="utf-8") if path.endswith(".gz") else open(path, encoding="utf-8")) as f:
        return [json.loads(l) for l in f if l.strip()]

def take(rows, n, seed=SEED):
    rows = list(rows); random.Random(seed).shuffle(rows)
    return rows if n is None else rows[:n]
print("all files ready")

## 6. Build the questions
Prompts use the same wording as the benchmark notebook, so the model learns the formats it is tested in — but the questions themselves are new.

In [ ]:
CODE_INSTRUCTION = ("Return the complete function implementation, including the signature and any "
                    "imports it needs, in a single ```python code block.")
LCB_SYSTEM = ("You are an expert Python programmer. You will be given a question (problem specification) and "
              "will generate a correct Python program that matches the specification and passes all tests.")
ITEMS = []

# ---- MBPP: drop the whole test split (ids 11-510) and every task that is in MBPP+
mbpp_plus_ids = {int(r["task_id"].split("/")[1]) for r in read_jsonl("MbppPlus.jsonl.gz")}
mbpp = [r for r in read_jsonl("mbpp.jsonl") if not (11 <= r["task_id"] <= 510) and r["task_id"] not in mbpp_plus_ids]
for r in take(mbpp, N_PER_DOMAIN["mbpp"]):
    names = re.findall(r"def\s+(\w+)\s*\(", r["code"])
    ep = next((n for n in names if re.search(rf"\b{n}\s*\(", r["test_list"][0])), None)
    if not ep:
        continue
    q = f"{r['text']}\n{r['test_list'][0]}\n\nThe function must be named `{ep}` and pass the test above.\n\n{CODE_INSTRUCTION}"
    ITEMS.append(dict(domain="mbpp", id=f"mbpp/{r['task_id']}", messages=[{"role": "user", "content": q}],
                      entry_point=ep, setup=r.get("test_setup_code", ""), tests=r["test_list"] + r.get("challenge_test_list", []),
                      reference=r["code"]))

# ---- APPS: stdin/stdout problems (no starter code), introductory + interview
def apps_question(text):
    return (f"### Question:\n{text}\n\n### Format: Read the inputs from stdin solve the problem and write the answer to "
            "the regular output (stdout). Do not directly test on the sample inputs. Enclose your code within delimiters "
            "as follows. Ensure that when the python program runs, it reads the inputs, runs the algorithm and writes "
            "output to STDOUT.\n```python\n# YOUR CODE HERE\n```\n\n### Answer: (use the provided format with backticks)\n\n")
apps = []
for r in read_jsonl("apps_train.jsonl"):
    if r["difficulty"] not in ("introductory", "interview") or r["starter_code"].strip():
        continue
    try:
        io_ = json.loads(r["input_output"]); sols = json.loads(r["solutions"])
    except Exception:
        continue
    if "fn_name" in io_ or not io_.get("inputs") or not all(isinstance(x, str) for x in io_["inputs"] + io_["outputs"]):
        continue
    if len(r["question"]) > 6000:
        continue
    apps.append((r, io_, sols))
for r, io_, sols in take(apps, N_PER_DOMAIN["apps"]):
    tests = list(zip(io_["inputs"], io_["outputs"]))[:8]
    ITEMS.append(dict(domain="apps", id=f"apps/{r['id']}", tests=tests, reference=sols[0] if sols else "",
                      messages=[{"role": "system", "content": LCB_SYSTEM}, {"role": "user", "content": apps_question(r["question"])}]))

# ---- GSM8K train
for i, r in take(list(enumerate(read_jsonl("gsm8k_train.jsonl"))), N_PER_DOMAIN["gsm8k"]):
    gold = float(r["answer"].rsplit("####", 1)[-1].strip().replace(",", ""))
    q = f"{r['question']}\n\nSolve the problem and put the final numeric answer inside \\boxed{{}}."
    ITEMS.append(dict(domain="gsm8k", id=f"gsm8k_train/{i}", gold=gold, messages=[{"role": "user", "content": q}]))

# ---- Instruction following: OASST2 English first-turn prompts + 1-3 checkable rules (IFEval rule types)
import nltk
for p in ("punkt", "punkt_tab"):
    nltk.download(p, quiet=True)
IFEVAL_DIR = os.path.join(WORK, "vendor", "instruction_following_eval")
os.makedirs(IFEVAL_DIR, exist_ok=True)
for f in ["instructions.py", "instructions_registry.py", "instructions_util.py", "evaluation_lib.py"]:
    if not os.path.exists(os.path.join(IFEVAL_DIR, f)):
        download(f"https://raw.githubusercontent.com/google-research/google-research/master/instruction_following_eval/{f}",
                 os.path.join(IFEVAL_DIR, f))
open(os.path.join(IFEVAL_DIR, "__init__.py"), "a").close()
sys.path.insert(0, os.path.dirname(IFEVAL_DIR))
from instruction_following_eval import evaluation_lib as ifeval_lib, instructions_registry as if_reg

SKIP_RULES = {"combination:repeat_prompt", "language:response_language"}   # need the prompt itself / non-English
RULES = [k for k in if_reg.INSTRUCTION_DICT if k not in SKIP_RULES]
random.seed(SEED)                       # IFEval picks random rule details (numbers, keywords) with `random`
_rng = random.Random(SEED)
oasst = [o for o in read_jsonl("oasst2_prompts.jsonl.gz")
         if o.get("lang") == "en" and o.get("parent_id") is None and 40 <= len(o["text"]) <= 600 and "```" not in o["text"]]
for o in take(oasst, N_PER_DOMAIN["if"]):
    ids = []
    for _ in range(50):
        if len(ids) >= _rng.choice([1, 2, 2, 3]): break
        c = _rng.choice(RULES)
        if c in ids or any(c in if_reg.INSTRUCTION_CONFLICTS.get(x, ()) or x in if_reg.INSTRUCTION_CONFLICTS.get(c, ()) for x in ids):
            continue
        ids.append(c)
    descs, kwargs = [], []
    for c in ids:
        inst = if_reg.INSTRUCTION_DICT[c](c)
        descs.append(inst.build_description())
        kwargs.append({k: v for k, v in (inst.get_instruction_args() or {}).items() if v is not None})
    prompt = o["text"].strip() + "\n\n" + " ".join(descs)
    ITEMS.append(dict(domain="if", id=f"if/{o['message_id']}", base=o["text"], rule_ids=ids, rule_kwargs=kwargs,
                      messages=[{"role": "user", "content": prompt}]))

# ---- Tool calls: Hermes function-calling (single turn); reference calls are parsed from its answers
def _load_call(s):
    s = s.strip()
    if s.startswith("\\n"): s = s[2:]
    for parse in (json.loads, ast.literal_eval):
        try:
            c = parse(s)
            if isinstance(c, dict) and "name" in c and isinstance(c.get("arguments"), dict):
                return {"name": c["name"], "arguments": c["arguments"]}
        except Exception:
            pass
    return None
hermes = []
for r in json.load(open(os.path.join(DATA_DIR, "hermes_fc_singleturn.json"), encoding="utf-8")):
    conv = r["conversations"]
    tools = json.loads(r["tools"]) if isinstance(r["tools"], str) else r["tools"]
    user = next((m["value"] for m in conv if m["from"] == "human"), None)
    raw = re.findall(r"<tool_call>\s*(.*?)\s*</tool_call>", conv[-1]["value"], re.DOTALL)
    calls = [_load_call(x) for x in raw]
    names = {t.get("function", {}).get("name") for t in tools} if isinstance(tools, list) else set()
    if not user or not tools or not calls or None in calls or not all(c["name"] in names for c in calls):
        continue
    hermes.append(dict(user=user, tools=tools, calls=calls, id=r["id"]))
for h in take(hermes, N_PER_DOMAIN["tools"]):
    ITEMS.append(dict(domain="tools", id=f"tools/{h['id']}", tools=h["tools"], ref_calls=h["calls"],
                      messages=[{"role": "user", "content": h["user"]}]))
# no_tool: real tools are offered, but the question has nothing to do with them -> answer without calling any
_used = {it["id"].split("/", 1)[1] for it in ITEMS if it["domain"] == "if"}   # keep the two sets apart
general_qs = take([o for o in oasst if "?" in o["text"] and o["message_id"] not in _used], N_PER_DOMAIN["no_tool"], seed=SEED + 1)
for o, h in zip(general_qs, take(hermes, N_PER_DOMAIN["no_tool"], seed=SEED + 2)):
    ITEMS.append(dict(domain="no_tool", id=f"no_tool/{o['message_id']}", tools=h["tools"],
                      messages=[{"role": "user", "content": o["text"].strip()}]))

import collections
print(collections.Counter(it["domain"] for it in ITEMS))
print("example:", ITEMS[0]["messages"][-1]["content"][:300])

## 7. Contamination check (no training question may look like a test question)
We collect every benchmark question (HumanEval+, MBPP+, LiveCodeBench v6, GSM8K test, IFEval, IFBench, BFCL) and drop any training question that shares a **13-word sequence** with one of them (the standard check used for GPT-3 and others).

In [ ]:
def grams(text, n=13):
    w = re.findall(r"\w+", text.lower())
    return {" ".join(w)} if len(w) < n else {" ".join(w[i:i + n]) for i in range(len(w) - n + 1)}

EVAL_TEXT = []
EVAL_TEXT += [r["prompt"] for r in read_jsonl("HumanEvalPlus.jsonl.gz")]
EVAL_TEXT += [r["prompt"] for r in read_jsonl("MbppPlus.jsonl.gz")]
EVAL_TEXT += [r["question"] for r in read_jsonl("gsm8k_test.jsonl")]
EVAL_TEXT += [r["prompt"] for r in read_jsonl("ifeval_input_data.jsonl")]
EVAL_TEXT += [r["prompt"] for r in read_jsonl("ifbench_test.jsonl")]
EVAL_TEXT += [r["question_content"] for r in read_jsonl("livecodebench_v6.jsonl")]
for f in EVAL_FILES:
    if f.startswith("BFCL"):
        for r in read_jsonl(f):
            EVAL_TEXT += [m["content"] for turn in r["question"] for m in turn if m.get("role") == "user"]
EVAL_GRAMS = set().union(*(grams(t) for t in EVAL_TEXT))
print(f"{len(EVAL_TEXT)} benchmark questions -> {len(EVAL_GRAMS):,} 13-word sequences")

def question_text(it):
    if it["domain"] == "if": return it["base"]                     # the rules are added by us
    return it["messages"][-1]["content"]

before = collections.Counter(it["domain"] for it in ITEMS)
ITEMS = [it for it in ITEMS if not (grams(question_text(it)) & EVAL_GRAMS)]
after = collections.Counter(it["domain"] for it in ITEMS)
print("removed for overlap:", {d: before[d] - after[d] for d in before})
print("kept:", dict(after), "| total", len(ITEMS))
assert len({it["id"] for it in ITEMS}) == len(ITEMS), "duplicate ids"
BY_ID = {it["id"]: it for it in ITEMS}

## 8. Checkers, and a self-test before any GPU time
Every answer is checked automatically. We first prove the checkers work: **reference answers must pass, wrong answers must fail.**

In [ ]:
def check(it, answer):
    d = it["domain"]
    if not answer.strip():
        return False
    if d == "mbpp":
        code = extract_code(answer, it["entry_point"])
        rc, _ = run_program((it["setup"] or "") + "\n" + code + "\n\n" + "\n".join(it["tests"]) + "\n", timeout=10)
        return rc == 0
    if d == "apps":
        code = extract_code(answer)
        for stdin, expected in it["tests"]:
            rc, out = run_program(code, stdin, timeout=4)
            if rc != 0 or not same_output(out, expected):
                return False
        return True
    if d == "gsm8k":
        pred = extract_final_number(answer)
        return pred is not None and abs(pred - it["gold"]) <= 1e-4 * max(1, abs(it["gold"]))
    if d == "if":
        prompt = it["messages"][-1]["content"]
        inp = ifeval_lib.InputExample(key=0, instruction_id_list=it["rule_ids"], prompt=prompt, kwargs=it["rule_kwargs"])
        return bool(ifeval_lib.test_instruction_following_strict(inp, {prompt: answer}).follow_all_instructions)
    if d == "tools":
        return same_calls(parse_tool_calls(answer, it["tools"]), it["ref_calls"])
    if d == "no_tool":
        return "<tool_call>" not in answer and len(answer.strip()) >= 20
    raise ValueError(d)

def check_many(pairs):
    # pairs: [(item, answer)] -> [bool]; programs run in parallel
    with ThreadPoolExecutor(max_workers=max(2, os.cpu_count() or 2)) as ex:
        return list(ex.map(lambda p: check(*p), pairs))

def reference(it, good=True):
    d = it["domain"]
    if d == "mbpp": return f"```python\n{it['reference']}\n```" if good else "```python\nprint('hi')\n```"
    if d == "apps": return f"```python\n{it['reference']}\n```" if good else "```python\nprint(0)\n```"
    if d == "gsm8k":
        g = it["gold"]; return f"\\boxed{{{int(g) if g == int(g) else g}}}" if good else "\\boxed{-999999}"
    if d == "tools": return render_tool_calls(it["ref_calls"]) if good else "I would call a tool here."
    if d == "no_tool":
        return "This question does not need any of the available tools; here is a direct answer." if good else \
               "<tool_call>\n<function=x>\n</function>\n</tool_call>"
    return None

print(f"{'domain':8s} {'reference':>10s} {'wrong':>7s}")
for d in ["mbpp", "apps", "gsm8k", "tools", "no_tool"]:
    its = [it for it in ITEMS if it["domain"] == d][:30]
    if not its: continue
    good = sum(check_many([(it, reference(it, True)) for it in its])) / len(its)
    bad = sum(check_many([(it, reference(it, False)) for it in its])) / len(its)
    print(f"{d:8s} {good:10.0%} {bad:7.0%}")
    assert bad == 0, f"{d}: wrong answers must fail"
    # APPS reference solutions are user-submitted and some are slow or wrong; the rest must pass
    assert good >= {"apps": 0.6, "tools": 0.9}.get(d, 0.95), f"{d}: reference answers should pass"
# IFEval rules: an empty-ish answer must fail, and the checker is the same Google code verified in the benchmark notebook
its = [it for it in ITEMS if it["domain"] == "if"][:30]
print(f"{'if':8s} {'(n/a)':>10s} {sum(check(it, 'ok') for it in its) / len(its):7.0%}  <- a one-word answer passing almost no rule sets")
print("\nCheckers OK.")

## 9. GPU worker (one per GPU)
Same design as the benchmark notebook, with one fix from the last run: instead of a fixed batch of 64 (which ran out of memory once answers got long), each GPU fills its batch up to a **token budget** (`BATCH_TOKENS`). If memory still runs out, the budget shrinks — and it **grows back** after a while, so one bad moment no longer slows the whole run.

In [ ]:
%%writefile gen_worker.py
import argparse, json, os, time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

ap = argparse.ArgumentParser()
ap.add_argument("--gpu", type=int, required=True)
ap.add_argument("--run_dir", required=True)
ap.add_argument("--model", required=True)
ap.add_argument("--batch", type=int, default=64)
ap.add_argument("--batch_tokens", type=int, default=80000)
ap.add_argument("--seed", type=int, default=0)
args = ap.parse_args()

def log(*a):
    print(time.strftime("%H:%M:%S"), f"[GPU{args.gpu}]", *a, flush=True)

dev = torch.device(f"cuda:{args.gpu}")
torch.cuda.set_device(dev)
torch.manual_seed(args.seed + 1000 * args.gpu)
dtype = torch.bfloat16 if torch.cuda.get_device_capability(dev)[0] >= 8 else torch.float16
tok = AutoTokenizer.from_pretrained(args.model, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(args.model, dtype=dtype, device_map={"": args.gpu},
                                             trust_remote_code=True).eval()
PAD = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
_eos = model.generation_config.eos_token_id
EOS = set(_eos if isinstance(_eos, list) else [_eos])
if "<|im_end|>" in tok.get_vocab():
    EOS.add(tok.convert_tokens_to_ids("<|im_end|>"))
EOS.discard(None)
DROP = [t for t in tok.all_special_tokens if t not in ("<think>", "</think>")]
log(f"model loaded ({dtype})")


@torch.no_grad()
def gen_chunk(seqs, steps, s):
    L = max(map(len, seqs))
    ids = torch.full((len(seqs), L), PAD, dtype=torch.long)
    mask = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, q in enumerate(seqs):
        ids[i, L - len(q):] = torch.tensor(q, dtype=torch.long)
        mask[i, L - len(q):] = 1
    sample = s["temperature"] > 0
    out = model.generate(input_ids=ids.to(dev), attention_mask=mask.to(dev), max_new_tokens=steps,
                         do_sample=sample, temperature=s["temperature"] if sample else None,
                         top_k=s["top_k"] if sample else None, top_p=s["top_p"] if sample else None,
                         repetition_penalty=s["repetition_penalty"], pad_token_id=PAD, eos_token_id=sorted(EOS))
    return out[:, L:].tolist()


run = args.run_dir
jobs = [json.loads(l) for l in open(os.path.join(run, "jobs.jsonl"))]
done = set()
for f in os.listdir(run):
    if f.startswith("answers_gpu") and f.endswith(".jsonl"):
        for l in open(os.path.join(run, f)):
            try:
                done.add(json.loads(l)["key"])
            except Exception:
                pass
claims = os.path.join(run, "claims")

def claim(i):
    try:
        os.close(os.open(os.path.join(claims, str(i)), os.O_CREAT | os.O_EXCL))
        return True
    except FileExistsError:
        return False

groups = {}
for j in jobs:
    if j["key"] not in done:
        groups.setdefault((j["sampling_name"], j["max_new_tokens"]), []).append(j["idx"])
order = sorted(groups, key=lambda g: (-g[1], -len(groups[g])))
out_f = open(os.path.join(run, f"answers_gpu{args.gpu}.jsonl"), "a")

def save(s, truncated):
    j = jobs[s["idx"]]
    text = tok.decode(s["gen"], skip_special_tokens=False)
    for t in DROP:
        text = text.replace(t, "")
    out_f.write(json.dumps({"key": j["key"], "text": text, "n_tokens": len(s["gen"]), "truncated": truncated,
                            "gpu": args.gpu}) + "\n")
    out_f.flush()

log(f"{len(done)} answers already saved, {sum(map(len, groups.values()))} to go")
budget, ok_streak = args.batch_tokens, 0
active, cur, n_done, n_tok, t_start = [], None, 0, 0, time.time()

def progress():
    with open(os.path.join(run, f"progress_gpu{args.gpu}.json"), "w") as f:
        json.dump({"start": t_start, "now": time.time(), "tokens": n_tok, "budget": budget,
                   "in_progress": sum(len(s["gen"]) for s in active)}, f)

while True:
    if not active:
        cur = next((g for g in order if groups[g]), None)
        if cur is None:
            break
    pend = groups[cur]
    while len(active) < args.batch and pend:
        i = pend.pop(0)
        if claim(i):
            active.append({"idx": i, "gen": []})
    if not active:
        continue
    max_new = cur[1]
    steps = min(max(256, max_new // 4), max(max_new - len(s["gen"]) for s in active))
    batch, used = [], 0                    # fill the batch up to the token budget
    for s in active:
        cost = len(jobs[s["idx"]]["prompt_ids"]) + len(s["gen"]) + steps
        if batch and used + cost > budget:
            break
        batch.append(s); used += cost
    remaining = [max_new - len(s["gen"]) for s in batch]
    t0 = time.perf_counter()
    try:
        new = gen_chunk([jobs[s["idx"]]["prompt_ids"] + s["gen"] for s in batch], steps, jobs[batch[0]["idx"]]["sampling"])
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(batch) == 1:
            s = batch[0]; active.remove(s); save(s, True); log("out of memory on a single answer: saved as cut off")
        else:
            budget, ok_streak = max(4000, int(used * 0.6)), 0
            log(f"out of memory -> token budget {budget}")
        continue
    secs = time.perf_counter() - t0
    ok_streak += 1
    if ok_streak >= 6 and budget < args.batch_tokens:      # memory was fine for a while: grow back
        budget, ok_streak = min(args.batch_tokens, int(budget * 1.25)), 0
    finished = []
    for s, toks, rem in zip(batch, new, remaining):
        toks = toks[:rem]
        stop = next((k for k, t in enumerate(toks) if t in EOS), None)
        n_tok += len(toks) if stop is None else stop
        if stop is not None:
            s["gen"] += toks[:stop]; finished.append((s, False))
        else:
            s["gen"] += toks
            if len(s["gen"]) >= max_new:
                finished.append((s, True))
    for s, truncated in finished:
        active.remove(s); save(s, truncated); n_done += 1
    progress()
    log(f"{cur[0]}/{cur[1]}: {len(batch)} in batch ({used} tokens), {steps} steps in {secs:.0f}s, {len(finished)} finished "
        f"(this GPU: {n_done}, {n_tok / (time.time() - t_start):.0f} tokens/s)")

progress()
out_f.close()
log("all done")

In [ ]:
from IPython.display import display, HTML

def run_jobs(name, jobs):
    # Runs the jobs on every GPU in the background and waits, showing live speed. Returns {key: answer record}.
    run_dir = os.path.join(GEN_DIR, name)
    os.makedirs(os.path.join(run_dir, "claims"), exist_ok=True)
    with open(os.path.join(run_dir, "jobs.jsonl"), "w") as f:
        f.writelines(json.dumps({**j, "idx": i}) + "\n" for i, j in enumerate(jobs))
    for c in os.listdir(os.path.join(run_dir, "claims")):
        os.remove(os.path.join(run_dir, "claims", c))

    def read():
        got = {}
        for f in os.listdir(run_dir):
            if f.startswith("answers_gpu"):
                for l in open(os.path.join(run_dir, f)):
                    try: r = json.loads(l)
                    except ValueError: continue
                    got.setdefault(r["key"], r)
        return got

    keys = {j["key"] for j in jobs}
    if keys <= set(read()):
        print(f"{name}: all {len(jobs)} answers already saved"); return read()
    procs = [subprocess.Popen([sys.executable, os.path.abspath("gen_worker.py"), "--gpu", str(g), "--run_dir", run_dir,
                               "--model", MODEL, "--batch", str(BATCH), "--batch_tokens", str(BATCH_TOKENS), "--seed", str(SEED)],
                              stdout=open(os.path.join(run_dir, f"worker_gpu{g}.log"), "a"), stderr=subprocess.STDOUT)
             for g in range(N_GPUS)]
    status, t0 = display(HTML("<pre>starting GPU workers...</pre>"), display_id=True), time.time()
    try:
        while True:
            got = read(); done = len(keys & set(got))
            prog = []
            for g in range(N_GPUS):
                try: prog.append(json.load(open(os.path.join(run_dir, f"progress_gpu{g}.json"))))
                except Exception: pass
            tps = sum(p["tokens"] / max(p["now"] - p["start"], 1) for p in prog) if prog else 0
            avg = (sum(got[k]["n_tokens"] for k in keys & set(got)) / done) if done else jobs[0]["max_new_tokens"] / 3
            todo = max((len(keys) - done) * avg - sum(p.get("in_progress", 0) for p in prog), 0)
            left = f"{todo / tps / 60:.0f} min" if tps > 0 else "measuring..."
            alive = sum(p.poll() is None for p in procs)
            status.update(HTML(f"<pre>{name}: {done}/{len(keys)} answers | {alive}/{N_GPUS} GPU workers | "
                               f"{(time.time() - t0) / 60:.0f} min | {tps:.0f} tokens/s | rough time left: {left}</pre>"))
            if done == len(keys):
                break
            if alive == 0:
                for g in range(N_GPUS):
                    print(f"----- worker_gpu{g}.log -----\n" + open(os.path.join(run_dir, f"worker_gpu{g}.log")).read()[-3000:])
                raise RuntimeError("GPU workers stopped early (logs above). Run this cell again to continue.")
            time.sleep(20)
    finally:
        for p in procs:
            if p.poll() is None: p.terminate()
    print(f"{name}: done in {(time.time() - t0) / 60:.0f} min")
    return read()
print("runner ready")

## 10. Pass A: every question, thinking OFF
Cheap first pass. Answers that are correct **and** have no hidden reasoning are perfect training targets already. If the model reasoned in secret first, we keep only the part after its `</think>` (if that part is correct).

In [ ]:
SAMPLING_A = {"gsm8k": "math"}            # everything else: Liquid's default
jobs_a = [dict(key=f"A|{it['id']}", sampling_name=SAMPLING_A.get(it["domain"], "general"),
               sampling=SAMPLING[SAMPLING_A.get(it["domain"], "general")], max_new_tokens=MAX_NEW_A,
               prompt_ids=tokenizer(render(it["messages"], False, it.get("tools")), add_special_tokens=False)["input_ids"])
          for it in ITEMS]
ANS_A = run_jobs("pass_A_thinking_off", jobs_a)

In [ ]:
CANDIDATES = []   # every generated answer, with its check result

def add_candidates(answers, pass_name, opened_think):
    pairs, recs = [], []
    for key, g in answers.items():
        item_id = key.split("|")[1]
        it = BY_ID.get(item_id)
        if it is None: continue
        reasoning, answer, status = split_thinking(g["text"], opened_think)
        recs.append(dict(id=item_id, domain=it["domain"], pass_=pass_name, key=key, answer=answer, status=status,
                         reasoning_tokens=n_tokens(reasoning), answer_tokens=n_tokens(answer),
                         total_tokens=g["n_tokens"], truncated=g["truncated"]))
        pairs.append((it, answer))
    for r, ok in zip(recs, check_many(pairs)):
        r["correct"] = bool(ok) and not r["truncated"]
    CANDIDATES.extend(recs)
    return recs

recs_a = add_candidates(ANS_A, "A", opened_think=False)
import pandas as pd
da = pd.DataFrame(recs_a)
da["hidden_reasoning"] = da["status"] == "closed"
print("Pass A (thinking OFF):")
display(da.groupby("domain").agg(n=("correct", "size"), correct=("correct", "mean"), hidden_reasoning=("hidden_reasoning", "mean"),
                                  cut_off=("truncated", "mean"), median_tokens=("total_tokens", "median"))
          .style.format({"correct": "{:.0%}", "hidden_reasoning": "{:.0%}", "cut_off": "{:.0%}", "median_tokens": "{:.0f}"}))
SOLVED_A = set(da[da.correct].id)
print(f"solved in pass A: {len(SOLVED_A)} / {len(ITEMS)}")

## 11. Pass B: thinking ON, only for what pass A could not solve
`TRIES_ON` tries each, at a slightly higher temperature so the tries differ. We keep only the **final answer** (after `</think>`).

In [ ]:
todo_b = [it for it in ITEMS if it["id"] not in SOLVED_A]
SAMPLING_B = {"gsm8k": "math"}
jobs_b = [dict(key=f"B|{it['id']}|{k}", sampling_name=SAMPLING_B.get(it["domain"], "diverse"),
               sampling=SAMPLING[SAMPLING_B.get(it["domain"], "diverse")], max_new_tokens=MAX_NEW_B,
               prompt_ids=tokenizer(render(it["messages"], True, it.get("tools")), add_special_tokens=False)["input_ids"])
          for it in todo_b for k in range(TRIES_ON)]
print(f"pass B: {len(todo_b)} questions x {TRIES_ON} tries = {len(jobs_b)} answers")
ANS_B = run_jobs("pass_B_thinking_on", jobs_b) if jobs_b else {}
recs_b = add_candidates(ANS_B, "B", opened_think=True)
if recs_b:
    db = pd.DataFrame(recs_b)
    print("Pass B (thinking ON):")
    display(db.groupby("domain").agg(answers=("correct", "size"), correct=("correct", "mean"), cut_off=("truncated", "mean"),
                                      median_thinking=("reasoning_tokens", "median"), median_final_answer=("answer_tokens", "median"))
              .style.format({"correct": "{:.0%}", "cut_off": "{:.0%}"}))

## 12. Build the training set
For each question: among its **correct** answers, keep the **shortest** clean final answer (no thinking tags, not cut off, at most `MAX_TARGET_TOKENS`). Each example is stored as chat messages (plus tools when there are any), ready for the fine-tuning notebook, which renders it with thinking OFF.

In [ ]:
def clean(a):
    return a.strip() and not any(t in a for t in ("<think>", "</think>", "<|im_start|>", "<|im_end|>"))

def source_of(c):
    if c["pass_"] == "B": return "thinking_on_final_answer"
    return "off_after_hidden_reasoning" if c["status"] == "closed" else "off_direct"

best = {}
for c in CANDIDATES:
    if c["correct"] and clean(c["answer"]) and c["answer_tokens"] <= MAX_TARGET_TOKENS:
        if c["id"] not in best or c["answer_tokens"] < best[c["id"]]["answer_tokens"]:
            best[c["id"]] = c

DATASET = []
for item_id, c in best.items():
    it = BY_ID[item_id]
    rec = dict(id=item_id, domain=it["domain"], source=source_of(c), target_tokens=c["answer_tokens"],
               messages=it["messages"] + [{"role": "assistant", "content": c["answer"]}])
    if it.get("tools"): rec["tools"] = it["tools"]
    DATASET.append(rec)
random.Random(SEED).shuffle(DATASET)
n_val = max(20, len(DATASET) // 20)
VAL, TRAIN = DATASET[:n_val], DATASET[n_val:]

ds = pd.DataFrame(DATASET)
print(f"Training examples: {len(TRAIN)}  |  validation: {len(VAL)}  |  questions with no correct answer: {len(ITEMS) - len(DATASET)}")
display(pd.crosstab(ds["domain"], ds["source"], margins=True))
display(ds.groupby("domain")["target_tokens"].describe()[["count", "mean", "50%", "max"]].round(0))

# how much shorter are the targets than thinking-ON answers to the same questions?
if recs_b:
    on_tok = db.groupby("id")["total_tokens"].mean()
    m = ds[ds.id.isin(on_tok.index)]
    if len(m):
        print(f"For questions solved via thinking ON: full ON answer {on_tok[m.id].mean():.0f} tokens "
              f"-> training target {m.target_tokens.mean():.0f} tokens ({on_tok[m.id].mean() / m.target_tokens.mean():.1f}x shorter)")

In [ ]:
# A few examples, one per domain
for d in ["mbpp", "apps", "gsm8k", "if", "tools", "no_tool"]:
    ex = next((r for r in TRAIN if r["domain"] == d), None)
    if ex:
        print(f"===== {d} ({ex['source']}, {ex['target_tokens']} tokens) =====")
        print("Q:", ex["messages"][-2]["content"][:400])
        print("A:", ex["messages"][-1]["content"][:600], "\n")

## 13. Save
- `instant_data/train.jsonl`, `instant_data/val.jsonl`: the training set (chat messages + tools)
- `instant_data/all_candidates.jsonl`: every generated answer with its check result (for transparency)
- `instant_data/stats.json`
- zipped to `instant_data.zip` (Output panel → download)

Optional: upload to the Hugging Face Hub as a dataset (set `PUSH_TO_HUB = True` and add a Kaggle secret `HF_TOKEN`).

In [ ]:
def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in rows)

write_jsonl(os.path.join(OUT_DIR, "train.jsonl"), TRAIN)
write_jsonl(os.path.join(OUT_DIR, "val.jsonl"), VAL)
write_jsonl(os.path.join(OUT_DIR, "all_candidates.jsonl"), CANDIDATES)
stats = dict(model=MODEL, questions=len(ITEMS), train=len(TRAIN), val=len(VAL),
             by_domain=ds["domain"].value_counts().to_dict(), by_source=ds["source"].value_counts().to_dict(),
             mean_target_tokens=float(ds["target_tokens"].mean()),
             settings=dict(N_PER_DOMAIN=N_PER_DOMAIN, TRIES_ON=TRIES_ON, MAX_NEW_A=MAX_NEW_A, MAX_NEW_B=MAX_NEW_B,
                           MAX_TARGET_TOKENS=MAX_TARGET_TOKENS, SAMPLING=SAMPLING, SEED=SEED))
json.dump(stats, open(os.path.join(OUT_DIR, "stats.json"), "w"), indent=1)
shutil.make_archive(os.path.join(WORK, "instant_data"), "zip", OUT_DIR)
print(json.dumps(stats, indent=1))
print("saved", os.path.join(WORK, "instant_data.zip"))

PUSH_TO_HUB = False
HUB_REPO = "Sanskar1724/instant-1.7b-sft-data"
if PUSH_TO_HUB:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import HfApi
    api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
    api.create_repo(HUB_REPO, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(folder_path=OUT_DIR, repo_id=HUB_REPO, repo_type="dataset")
    print("uploaded to", HUB_REPO, "(private)")